# 04 — End-to-End Code Generation Pipeline Demo

Demonstrates the complete HaluGuard pipeline on individual and batched RepoBench examples.

**Pipeline stages:**
```
RepoBench example
  ↓
  CodeBERT embedding (query + chunks)
  ↓
  HCCS scoring (best trained model)
  ↓
  Type-router pre-emptive boost
  ↓
  DeepSeek-Coder generation (top-5 chunks)
  ↓
  Sandbox execution (execute_code)
  ↓  [if failed]
  Error → type_router boost → re-rank → retry (EFL, max 3x)
  ↓
  Metrics: Exact Match, Edit Similarity
```

**Requires:**
- Notebook 01: embeddings computed
- Notebook 02: best checkpoint saved to `checkpoints/`
- Notebook 03 (optional): for comparison with baselines

## 0. Setup

In [ ]:
import subprocess, sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/HaluGuard'
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR
except ImportError:
    DRIVE_ROOT = None
    REPO_DIR = os.path.abspath('..')

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

In [ ]:
import json, time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

from haluguard.models import MODEL_REGISTRY, build_model
from haluguard.hccs import HCCSScorer, embed_code, batch_embed
from haluguard.efl import run_efl, execute_code, build_completion_prompt as efl_prompt
from haluguard.generate import generate_next_line, build_completion_prompt
from haluguard.type_router import predict_boost, boost_scores
from haluguard.baselines import cosine_scores, bm25_select, no_context_select
from haluguard.evaluate import exact_match, edit_similarity, compute_metrics, compute_retrieval_summary

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Models

In [ ]:
BASE_DIR    = Path(REPO_DIR)
EMB_DIR     = BASE_DIR / 'data' / 'embeddings'
CKPT_DIR    = Path(DRIVE_ROOT) / 'checkpoints' if DRIVE_ROOT else BASE_DIR / 'checkpoints'
RESULTS_DIR = BASE_DIR / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load CodeBERT encoder ────────────────────────────────────────────────────
ENCODER_NAME = 'microsoft/codebert-base'
cb_tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
cb_encoder   = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
cb_encoder.eval()
print('CodeBERT loaded')

# ── Load best HCCS scorer ────────────────────────────────────────────────────
# Read metadata to find the best checkpoint
meta_path = CKPT_DIR / 'best_model_meta.json'
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    best_model_name = meta['best_model']
    print(f'Best model from metadata: {best_model_name}  (test MRR={meta.get("test_mrr", "?"):.4f})')
else:
    best_model_name = 'interaction_mlp'   # fallback
    print(f'No metadata found — using fallback: {best_model_name}')

# Try to load the new-style model checkpoint
ckpt_path = CKPT_DIR / f'{best_model_name}_best.pt'
if ckpt_path.exists() and best_model_name in MODEL_REGISTRY:
    from haluguard.models import InteractionMLP, DualEncoder, BilinearScorer
    # Instantiate correct class and load weights
    hccs_scorer = build_model(best_model_name)
    hccs_scorer.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    hccs_scorer = hccs_scorer.to(DEVICE).eval()
    print(f'Loaded {best_model_name} scorer from {ckpt_path.name}')
else:
    # Fall back to legacy HCCSScorer
    legacy_ckpt = next((CKPT_DIR / f for f in ['hccs_unixcoder_last3_best.pt',
                                                 'hccs_codebert_full_best.pt']
                         if (CKPT_DIR / f).exists()), None)
    if legacy_ckpt:
        hccs_scorer = HCCSScorer.load(legacy_ckpt).to(DEVICE).eval()
        print(f'Loaded legacy HCCSScorer from {legacy_ckpt.name}')
    else:
        raise FileNotFoundError('No HCCS checkpoint found. Run notebook 02 first.')

In [ ]:
# ── Load DeepSeek-Coder ──────────────────────────────────────────────────────
GEN_MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-base'
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME, dtype=torch.float16, device_map='auto'
)
gen_model.eval()
print(f'DeepSeek-Coder loaded on {next(gen_model.parameters()).device}')

def gen_fn(prompt: str) -> str:
    """Generate a single next line."""
    return generate_next_line(
        prompt, gen_tokenizer, gen_model,
        device=DEVICE, max_new_tokens=64, temperature=0.2,
    )

## 2. Load Pre-computed Embeddings + Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('tianyang/repobench_python_v1.1', split='cross_file_first')
print(f'RepoBench: {len(ds)} examples')

# Load pre-computed embeddings (same as used in training)
def _try_paths(*paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f'None found: {paths}')

query_embs_path = _try_paths(
    EMB_DIR / 'query_embeddings__codebert__last3.pt',
    EMB_DIR / 'query_embeddings.pt',
)
chunk_embs_path = _try_paths(
    EMB_DIR / 'chunk_embeddings__codebert.pt',
    EMB_DIR / 'chunk_embeddings.pt',
)

query_embs   = torch.load(query_embs_path, map_location='cpu').float()
chunk_embs   = torch.load(chunk_embs_path, map_location='cpu')
gold_indices = torch.load(EMB_DIR / 'gold_indices.pt', map_location='cpu')
test_indices = torch.load(EMB_DIR / 'test_indices.pt', map_location='cpu') \
               if (EMB_DIR / 'test_indices.pt').exists() else list(range(100))

if isinstance(gold_indices, torch.Tensor):
    gold_indices = gold_indices.tolist()

print(f'Query embs : {query_embs.shape}')
print(f'Test examples: {len(test_indices)}')

## 3. Single Example — Step-by-Step Walkthrough

In [ ]:
# Pick one test example to walk through in detail
DEMO_IDX = test_indices[0]
ex = ds[DEMO_IDX]

print(f'Example index: {DEMO_IDX}')
print(f'Gold snippet index: {ex["gold_snippet_index"]}')
print(f'Number of context chunks: {len(ex["context"])}')
print(f'Ground truth next line: {ex["next_line"]!r}')
print(f'\nLast 3 lines of cropped_code:')
for line in ex['cropped_code'].splitlines()[-3:]:
    print(f'  {line}')

In [ ]:
# ── Step 1: Embed query + chunks ─────────────────────────────────────────────
q_emb  = query_embs[DEMO_IDX].numpy()     # pre-computed
c_embs = chunk_embs[DEMO_IDX].numpy()     # pre-computed
n_chunks = c_embs.shape[0]
print(f'Query embedding: {q_emb.shape}')
print(f'Chunk embeddings: {c_embs.shape}  ({n_chunks} chunks)')

# ── Step 2: HCCS scoring ─────────────────────────────────────────────────────
if hasattr(hccs_scorer, 'score'):
    # New-style model (haluguard.models)
    q_t = torch.tensor(q_emb, dtype=torch.float32).to(DEVICE)
    c_t = torch.tensor(c_embs, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        hccs_raw = hccs_scorer.score(q_t, c_t).cpu().numpy()
else:
    # Legacy HCCSScorer
    hccs_raw = hccs_scorer.score_chunks(q_emb, c_embs, device=DEVICE)

# ── Step 3: Type-router pre-emptive boost ────────────────────────────────────
boosts = predict_boost(ex['cropped_code'])
hccs_boosted = boost_scores(hccs_raw, ex['context'], boosts)

# ── Step 4: Select top-5 chunks ──────────────────────────────────────────────
TOP_K = 5
top_indices_hccs   = list(np.argsort(hccs_boosted)[::-1][:TOP_K])
top_indices_cosine = list(np.argsort(cosine_scores(q_emb, c_embs))[::-1][:TOP_K])
gold_index = int(ex['gold_snippet_index'])

print(f'\nGold chunk index : {gold_index}')
print(f'HCCS top-5       : {top_indices_hccs}  (gold in top-5: {gold_index in top_indices_hccs})')
print(f'Cosine top-5     : {top_indices_cosine}  (gold in top-5: {gold_index in top_indices_cosine})')
print(f'\nDetected boosts  : {boosts}')

In [ ]:
# ── Step 5: Generate next line (HCCS context) ────────────────────────────────
snippets_hccs = [ex['context'][j]['snippet'] for j in top_indices_hccs]
prompt_hccs   = build_completion_prompt(ex['cropped_code'], ex['import_statement'], snippets_hccs)

print(f'Prompt length: {len(prompt_hccs)} chars')
pred_hccs = gen_fn(prompt_hccs)

print(f'\n── Generation Result ──')
print(f'Predicted : {pred_hccs!r}')
print(f'Ground truth: {ex["next_line"]!r}')
print(f'EM: {exact_match(pred_hccs, ex["next_line"]):.0f}')
print(f'ES: {edit_similarity(pred_hccs, ex["next_line"]):.3f}')

In [ ]:
# ── Step 6: Execution Feedback Loop (EFL) ───────────────────────────────────
# Run EFL — generate, execute in sandbox, retry on failure with error-guided re-ranking

efl_result = run_efl(
    cropped_code=ex['cropped_code'],
    import_statement=ex['import_statement'],
    contexts=ex['context'],
    scores=hccs_boosted,
    generate_fn=gen_fn,
    top_k=TOP_K,
    max_iterations=3,
    timeout=10,
)

print('── EFL Result ──')
print(f'Best prediction: {efl_result.code!r}')
print(f'Passed sandbox : {efl_result.passed}')
print(f'EFL iterations : {efl_result.iterations}')
print(f'EM (EFL)       : {exact_match(efl_result.code, ex["next_line"]):.0f}')
print(f'ES (EFL)       : {edit_similarity(efl_result.code, ex["next_line"]):.3f}')

if len(efl_result.history) > 1:
    print('\nEFL history:')
    for i, hr in enumerate(efl_result.history):
        print(f'  Iter {i}: passed={hr.passed}  error={hr.error_type}')

## 4. Batch Evaluation — HCCS vs Baselines (100 Examples)

In [ ]:
N_BATCH = min(100, len(test_indices))   # increase to len(test_indices) for full eval
batch_indices = test_indices[:N_BATCH]

methods = {
    'no_context':    [],
    'bm25':          [],
    'cosine':        [],
    'hccs':          [],
    'hccs_efl':      [],
}

for step, idx in enumerate(tqdm(batch_indices, desc='Batch eval')):
    ex = ds[idx]
    q_emb  = query_embs[idx].numpy()
    c_embs = chunk_embs[idx].numpy()
    gt     = ex['next_line']

    # No context
    prompt = build_completion_prompt(ex['cropped_code'], ex['import_statement'], [])
    methods['no_context'].append((gen_fn(prompt), gt))

    # BM25
    bm25_idx = bm25_select(ex['cropped_code'], ex['context'], top_k=TOP_K)
    snippets = [ex['context'][j]['snippet'] for j in bm25_idx]
    prompt = build_completion_prompt(ex['cropped_code'], ex['import_statement'], snippets)
    methods['bm25'].append((gen_fn(prompt), gt))

    # Cosine
    cos_idx = list(np.argsort(cosine_scores(q_emb, c_embs))[::-1][:TOP_K])
    snippets = [ex['context'][j]['snippet'] for j in cos_idx]
    prompt = build_completion_prompt(ex['cropped_code'], ex['import_statement'], snippets)
    methods['cosine'].append((gen_fn(prompt), gt))

    # HCCS (no EFL)
    if hasattr(hccs_scorer, 'score'):
        q_t = torch.tensor(q_emb).to(DEVICE)
        c_t = torch.tensor(c_embs).to(DEVICE)
        with torch.no_grad():
            raw_scores = hccs_scorer.score(q_t, c_t).cpu().numpy()
    else:
        raw_scores = hccs_scorer.score_chunks(q_emb, c_embs, device=DEVICE)
    boosted = boost_scores(raw_scores, ex['context'], predict_boost(ex['cropped_code']))
    hccs_idx = list(np.argsort(boosted)[::-1][:TOP_K])
    snippets = [ex['context'][j]['snippet'] for j in hccs_idx]
    prompt = build_completion_prompt(ex['cropped_code'], ex['import_statement'], snippets)
    methods['hccs'].append((gen_fn(prompt), gt))

    # HCCS + EFL
    efl_res = run_efl(
        ex['cropped_code'], ex['import_statement'], ex['context'],
        boosted, gen_fn, top_k=TOP_K, max_iterations=3, timeout=10,
    )
    methods['hccs_efl'].append((efl_res.code, gt))

print(f'Batch eval complete ({N_BATCH} examples)')

In [ ]:
# Results table
print(f'\n── Batch Results (n={N_BATCH}) — RepoBench ──')
print(f'{"Method":<18} {"EM":>7} {"Edit Sim":>9}')
print('-' * 38)

batch_results = []
for method, pairs in methods.items():
    preds = [p for p, _ in pairs]
    refs  = [r for _, r in pairs]
    m = compute_metrics(preds, refs)
    batch_results.append({'method': method, **m})
    print(f'{method:<18} {m["em"]:>7.3f} {m["es"]:>9.3f}')

with open(RESULTS_DIR / 'pipeline_demo_results.json', 'w') as f:
    json.dump(batch_results, f, indent=2)
print(f'\nSaved to {RESULTS_DIR}/pipeline_demo_results.json')

## 5. Error Analysis — Where Does HCCS Help vs Hurt?

In [ ]:
# Compare HCCS vs cosine on same examples — show cases where one wins
from haluguard.hccs import HallucinationType

hccs_wins, cosine_wins, ties = [], [], []

for step, idx in enumerate(batch_indices):
    ex = ds[idx]
    pred_hccs   = methods['hccs'][step][0]
    pred_cosine = methods['cosine'][step][0]
    gt          = ex['next_line']

    em_hccs   = exact_match(pred_hccs, gt)
    em_cosine = exact_match(pred_cosine, gt)

    if em_hccs > em_cosine:
        hccs_wins.append({'idx': idx, 'gt': gt, 'hccs': pred_hccs, 'cosine': pred_cosine})
    elif em_cosine > em_hccs:
        cosine_wins.append({'idx': idx, 'gt': gt, 'hccs': pred_hccs, 'cosine': pred_cosine})
    else:
        ties.append(idx)

print(f'HCCS wins  : {len(hccs_wins):>3}')
print(f'Cosine wins: {len(cosine_wins):>3}')
print(f'Ties       : {len(ties):>3}')

# Show first 3 cases where HCCS wins
print('\n── HCCS wins (first 3) ──')
for w in hccs_wins[:3]:
    ex_w = ds[w['idx']]
    print(f"  GT     : {w['gt']!r}")
    print(f"  HCCS   : {w['hccs']!r}")
    print(f"  Cosine : {w['cosine']!r}")
    print()